# 01 EDA
Run `python -m src.data_loader download` first. Outputs are not stored; run the cells to generate them.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, plotly.express as px
from src import config
from src.data_loader import load_raw_data
from src.preprocessing import rename_columns, clean_data
raw = rename_columns(load_raw_data())
df, report = clean_data(raw)
print(df.shape); print(report)
df.dtypes

In [ ]:
df.describe().T

## Target analysis

In [ ]:
counts = df['machine_failure'].value_counts()
print(counts, (counts / len(df)).round(4))
px.bar(counts.reset_index(), x='machine_failure', y='count', title='Failure distribution')

## Sensor distributions and failure vs sensor values

In [ ]:
sensors = ['air_temperature_k','process_temperature_k','rotational_speed_rpm','torque_nm','tool_wear_min']
for s in sensors:
    px.histogram(df, x=s, color='machine_failure', barmode='overlay', nbins=50, title=s).show()

In [ ]:
px.imshow(df[sensors + ['machine_failure']].corr().round(2), text_auto=True, title='Correlation matrix')

## Outliers (IQR rule)
The dataset has no time axis, so sensor 'trends' are shown as distributions by failure outcome. Vibration, pressure, voltage and current do not exist in AI4I 2020.

In [ ]:
for s in sensors:
    q1, q3 = df[s].quantile([.25, .75]); iqr = q3 - q1
    n = ((df[s] < q1 - 1.5*iqr) | (df[s] > q3 + 1.5*iqr)).sum()
    print(f'{s}: {n} IQR outliers')
px.box(df, y=sensors[3], x='machine_failure', points='outliers')